In [ ]:
#step1:-> import pd,np,os
import pandas as pd
import numpy as np
import os
from decimal import Decimal, ROUND_HALF_UP

In [ ]:
#step2:-> load data
from google.colab import files
files.upload()

In [ ]:
# step3:-> read all csv
df_daily = pd.read_csv("store_daily.csv")
df_inventory = pd.read_csv("inventory.csv")
df_feedback = pd.read_csv("feedback.csv")

print("Original shapes:", df_daily.shape, df_inventory.shape, df_feedback.shape)

Data Quality Report & Cleaning (store_daily.csv)


In [ ]:
# step4:-> clean store_daily data (Duplicates, Negatives, IQR Capping, Imputation)
os.makedirs("data", exist_ok=True)
os.makedirs("analysis", exist_ok=True)

df = df_daily.copy()

# 1. Remove exact duplicate rows
initial_len = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
duplicates_removed = initial_len - len(df)

log_rows = []

# 2. Treat negative footfall as missing
neg_mask = df["footfall"] < 0
for idx in df[neg_mask].index:
    row_data = df.loc[idx]
    log_rows.append({
        "store": row_data["store"],
        "date": row_data["date"],
        "old_footfall": row_data["footfall"],
        "new_footfall": "",
        "rule": "negative_to_missing"
    })
df.loc[neg_mask, "footfall"] = np.nan
df["footfall"] = pd.to_numeric(df["footfall"], errors="coerce")

# 3. IQR bounds and capping per store
stores = df["store"].unique()
for store in stores:
    s_mask = df["store"] == store
    s_vals = df.loc[s_mask, "footfall"].dropna()
    q1 = s_vals.quantile(0.25)
    q3 = s_vals.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    upper_mask = s_mask & (df["footfall"] > upper)
    for idx in df[upper_mask].index:
        old_val = df.loc[idx, "footfall"]
        # Round half-up to 2 decimals
        capped_val = float(Decimal(str(upper)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))
        df.loc[idx, "footfall"] = capped_val
        log_rows.append({
            "store": store,
            "date": df.loc[idx, "date"],
            "old_footfall": old_val,
            "new_footfall": capped_val,
            "rule": "iqr_cap"
        })

# 4. Median imputation per store
for store in stores:
    s_mask = df["store"] == store
    s_median = df.loc[s_mask, "footfall"].median()
    missing_mask = s_mask & df["footfall"].isna()
    for idx in df[missing_mask].index:
        df.loc[idx, "footfall"] = s_median
        log_rows.append({
            "store": store,
            "date": df.loc[idx, "date"],
            "old_footfall": "",
            "new_footfall": s_median,
            "rule": "median_impute"
        })

# 5.feature columns
df["revenue_per_footfall"] = round(df["revenue_inr"] / df["footfall"], 2)
df["marketing_efficiency"] = round(df["revenue_inr"] / df["marketing_spend_inr"], 2)
df["achievement_pct"] = round((df["revenue_inr"] / df["daily_target_inr"]) * 100, 2)
df["shortfall_inr"] = round(df["daily_target_inr"] - df["revenue_inr"], 2)

dt_series = pd.to_datetime(df["date"])
df["weekend_flag"] = dt_series.dt.dayofweek.apply(lambda x: "Yes" if x >= 5 else "No")
df["performance_band"] = df["achievement_pct"].apply(lambda ach: "Excellent" if ach >= 100 else ("Good" if ach >= 75 else "Poor"))

# Save cleaned file and cleaning log
df.to_csv("data/store_daily_clean.csv", index=False)
log_df = pd.DataFrame(log_rows)
log_df.to_csv("analysis/cleaning_log.csv", index=False)

print("Cleaning completed! Cleaned shape:", df.shape)
print("Duplicates removed:", duplicates_removed)